# Fase 06 — Segmentação de mercado e perfil físico

**Pergunta:** como o mercado se segmenta por preço e por perfil físico?

**Hipótese:** banda de mercado (preço) e cluster de perfil físico (sem price/lat/long/zipcode) capturam
mecanismos distintos (P3).

**Nota (pós fase 12):** a fase 05 havia adotado augmentation (Técnica B); a fase 12 reverteu essa
decisão após o Error Matrix (fase 11) mostrar piora em `test`. Esta fase roda sobre o `train` **sem**
augmentation (15.100 linhas, o mesmo split canônico da fase 03) — números abaixo refletem o pipeline
corrigido. `train` vem sempre de `data/trusted/train_for_pipeline.parquet` (artefato da fase 05,
conforme `configs/augmentation.yaml`), nunca do split bruto diretamente; `test`/`val` vêm do split
bruto, sempre reais.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
import json
from pathlib import Path

from src.scripts.run_segmentation import run_segmentation_pipeline

report = run_segmentation_pipeline()

Path("reports/segmentation_report.json").write_text(json.dumps(report, indent=2, default=str))
print("best_k:", report["best_k"])
print("price_band_counts:", report["price_band_counts"])
print("property_cluster_counts:", report["property_cluster_counts"])

best_k: 3
price_band_counts: {'Standard': 5940, 'Premium': 5496, 'Entry': 5259, 'Luxury': 4917}
property_cluster_counts: {0: 11651, 1: 9798, 2: 163}


In [3]:
import pandas as pd

df = pd.read_parquet("data/trusted/house_segments.parquet")
print("house_segments.parquet shape:", df.shape)
display(df[["id", "price_log", "price_quartile", "price_band", "property_cluster"]].head())

k_scan_df = pd.DataFrame(report["k_scan"])
display(k_scan_df)

separation_df = pd.DataFrame(report["price_band_separation"])
display(separation_df)

cluster_profile_df = pd.DataFrame(report["property_cluster_profile_median"]).T
display(cluster_profile_df)

house_segments.parquet shape: (21612, 53)


,id,price_log,price_quartile,price_band,property_cluster
0,7129300520,12.309987,Q1,Entry,0
1,6414100192,13.195616,Q3,Premium,1
2,1954400510,13.142168,Q3,Premium,0
3,7237550310,14.018452,Q4,Luxury,1
4,1321400060,12.458779,Q1,Entry,1


,k,silhouette
0,2,0.272233
1,3,0.282586
2,4,0.233889
3,5,0.225012
4,6,0.229558


,grade,sqft_living,waterfront_rate,view_rate
Q1,7.0,1430.0,0.000000,0.024901
Q2,7.0,1760.0,0.000788,0.103204
Q3,8.0,2030.0,0.002940,0.189471
Q4,9.0,2980.0,0.026225,0.622517


,sqft_living,sqft_lot,bedrooms,bathrooms,floors,grade,condition,view,waterfront,property_age
0,1540.0,7650.0,3.0,1.75,1.0,7.0,3.0,0.0,0.0,58.0
1,2530.0,7488.5,4.0,2.50,2.0,8.0,3.0,0.0,0.0,17.0
2,2850.0,17342.0,3.0,2.50,2.0,9.0,3.0,4.0,1.0,54.0


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
order = ["Entry", "Standard", "Premium", "Luxury"]
counts = df["price_band"].value_counts().reindex(order)
axes[0].bar(order, counts.values, color="#4C72B0")
axes[0].set_title("contagem por banda de preco")

axes[1].plot(k_scan_df["k"], k_scan_df["silhouette"], marker="o", color="#DD8452")
axes[1].axvline(report["best_k"], color="gray", linestyle="--")
axes[1].set_title(f"silhouette por k (k={report['best_k']})")

cluster_counts = df["property_cluster"].value_counts().sort_index()
axes[2].bar(cluster_counts.index.astype(str), cluster_counts.values, color="#55A868")
axes[2].set_title("contagem por cluster fisico")
plt.tight_layout()
plt.savefig("reports/figures/06_segmentation.png", dpi=110)
plt.close()
print(report["price_band_counts"])
print(report["property_cluster_counts"])

{'Standard': 5940, 'Premium': 5496, 'Entry': 5259, 'Luxury': 4917}
{0: 11651, 1: 9798, 2: 163}


## Análise

Banda de preço (quartis de `price_log`, fit em `train`): `Entry` 5.259, `Standard` 5.940, `Premium`
5.496, `Luxury` 4.917. Cluster físico k=3 (silhouette): antigo/compacto (11.651, `sqft_living` mediano
1.540, idade mediana 58 anos), moderno/grande (9.798, `sqft_living` mediano 2.530, idade mediana 17
anos), luxo físico raro (163, `sqft_living` mediano 2.850, `waterfront`/`view` altos). Números
idênticos aos da execução original (antes da reestruturação), como esperado — sem augmentation, o
`train` é exatamente o mesmo dataset.

![segmentação](../figures/06_segmentation.png)

O scan de k (2 a 6) confirma k=3 como o melhor por silhouette — os clusters resultantes têm leitura
física direta (tamanho/idade/grade), separando um segmento "luxo físico" pequeno e raro (163 imóveis,
`waterfront`/`view` altos) dos dois clusters majoritários (antigo/compacto vs. moderno/grande). Bandas
de preço e clusters físicos capturam eixos distintos: a correlação entre `price_band` e
`property_cluster` não é 1:1 (por exemplo, o cluster "antigo/compacto" ainda cobre imóveis em várias
bandas de preço), confirmando a hipótese P3.

## Decisão

`data/trusted/house_segments.parquet` (corrigido, sem augmentation) pronto para as fases 07-08.

## Artefatos

`data/trusted/house_segments.parquet`, `data/trusted/price_quartiles.json`,
`artifacts/property_cluster_model.pkl`, `reports/segmentation_report.json`,
`reports/figures/06_segmentation.png`

# Extensão exploratória (ad-hoc, fora do pipeline formal)

A fase 03 mostrou que o split é um **holdout geográfico completo por zipcode** (train: 49 zips, test:
10, val: 11, interseção = 0 — ver `notebooks/03_canonical_split.ipynb`). `price_band` e
`property_cluster` são fitados só em `train` (P1) e aplicados em `test`/`val` — mas a fase 06 original
nunca confere se essa aplicação se sustenta em zipcodes 100% novos, nem se os segmentos batem com o
erro real do modelo (medido na fase 04). Duas extensões, só leitura, sem retreinar `price_bands`/
`property_clusters`:

1. **Estabilidade fora de grupo:** `price_band`/`property_cluster` foram fitados em 49 zipcodes — será
   que a distribuição se mantém nos 10 zipcodes de `test`, nunca vistos no fit? Segmento raro (cluster
   "luxo físico", 163/15.100 = 1,08% de `train`) é o mais exposto a sumir ou disparar por causa de
   *qual* zipcode caiu em qual partição, não do critério de cluster em si.
2. **Cruzamento com erro real:** `property_cluster` é construído exatamente nas features que a extensão
   da fase 04 apontou como as que mais correlacionam com erro (`sqft_living`, `bathrooms`, `grade`,
   `bedrooms`...). Reusamos o mesmo modelo diagnóstico da fase 04 (`BASELINE_FEATURES`,
   `ABLATION_MODEL_PARAMS`) pra testar se o cluster raro é, de fato, o grupo de maior erro — o que
   decidiria se `property_cluster`/distância ao centróide vale a pena virar feature real (fases 07-08),
   não só rótulo descritivo.

In [5]:
segments = pd.read_parquet("data/trusted/house_segments.parquet")

band_order = ["Entry", "Standard", "Premium", "Luxury"]
band_pct = (segments.groupby("split")["price_band"]
            .value_counts(normalize=True).unstack().reindex(columns=band_order) * 100)
cluster_pct = (segments.groupby("split")["property_cluster"]
               .value_counts(normalize=True).unstack() * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
band_pct.reindex(["train", "test", "val"]).plot(kind="bar", ax=axes[0],
                                                 color=["#55A868", "#4C72B0", "#DD8452", "#C44E52"])
axes[0].set_title("% price_band por particao (limiares fitados so em train)")
axes[0].set_ylabel("% da particao")
axes[0].legend(title="price_band")

cluster_pct.reindex(["train", "test", "val"]).plot(kind="bar", ax=axes[1],
                                                     color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("% property_cluster por particao (KMeans fitado so em train)")
axes[1].set_ylabel("% da particao")
axes[1].legend(title="property_cluster")
plt.tight_layout()
plt.savefig("reports/figures/06_segment_stability_out_of_group.png", dpi=110)
plt.close()

print("price_band, % por particao:")
print(band_pct.reindex(["train", "test", "val"]).round(2))
print("\nproperty_cluster, % por particao:")
print(cluster_pct.reindex(["train", "test", "val"]).round(2))
print("\nn absoluto do cluster raro (2, 'luxo fisico') por particao:")
print(segments[segments["property_cluster"] == 2].groupby("split").size())
print("\nsaved reports/figures/06_segment_stability_out_of_group.png")

price_band, % por particao:
price_band  Entry  Standard  Premium  Luxury
split                                       
train       25.00     25.22    24.78   25.00
test        29.16     31.05    21.56   18.23
val         18.43     33.89    30.61   17.06

property_cluster, % por particao:
property_cluster      0      1     2
split                               
train             51.68  47.57  0.75
test              56.16  42.40  1.44
val               61.07  38.62  0.31

n absoluto do cluster raro (2, 'luxo fisico') por particao:
split
test      38
train    113
val       12
dtype: int64

saved reports/figures/06_segment_stability_out_of_group.png


## Extensão 2 — `property_cluster`/`price_band` batem com o erro real?

Reusa o mesmo modelo diagnóstico da fase 04 (`BASELINE_FEATURES`, `ABLATION_MODEL_PARAMS`, treinado em
`train` raw, sem segmentação) pra gerar erro absoluto por imóvel de `test`, e cruza com o `property_cluster`/
`price_band` que a fase 06 atribuiu a cada um (fit só em `train`, aplicado em `test`).

In [ ]:
import numpy as np
import seaborn as sns
from xgboost import XGBRegressor

from src.scripts.analyze_geographic_coverage import BASELINE_FEATURES
from src.validation.ablation import ABLATION_MODEL_PARAMS

house = pd.read_parquet("data/processed/house_clean.parquet")
split_raw = pd.read_parquet("data/processed/split_assignment.parquet")
df_raw = house.merge(split_raw[["id", "date", "split"]], on=["id", "date"], how="left")
train_raw = df_raw[df_raw["split"] == "train"].reset_index(drop=True)
test_raw = df_raw[df_raw["split"] == "test"].reset_index(drop=True)

model = XGBRegressor(**ABLATION_MODEL_PARAMS, n_jobs=-1)
model.fit(train_raw[BASELINE_FEATURES], train_raw["price_log"])
pred_log = model.predict(test_raw[BASELINE_FEATURES])
abs_error = np.abs(np.expm1(test_raw["price_log"]) - np.expm1(pred_log))

test_err = test_raw[["id", "date"]].copy()
test_err["abs_error"] = abs_error.values
test_err = test_err.merge(
    segments[segments["split"] == "test"][["id", "date", "property_cluster", "price_band"]],
    on=["id", "date"], how="left",
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.boxplot(data=test_err, x="property_cluster", y="abs_error", ax=axes[0],
            hue="property_cluster", legend=False,
            palette={0: "#4C72B0", 1: "#DD8452", 2: "#C44E52"})
axes[0].set_title("erro em test, por property_cluster (fitado em train)")

sns.boxplot(data=test_err, x="price_band", y="abs_error", order=band_order, ax=axes[1],
            hue="price_band", legend=False,
            palette={"Entry": "#55A868", "Standard": "#4C72B0", "Premium": "#DD8452", "Luxury": "#C44E52"})
axes[1].set_title("erro em test, por price_band (fitado em train)")
plt.tight_layout()
plt.savefig("reports/figures/06_error_by_segment.png", dpi=110)
plt.close()

print("erro por property_cluster:")
print(test_err.groupby("property_cluster")["abs_error"].agg(n="count", median="median", mean="mean"))
print("\nerro por price_band:")
print(test_err.groupby("price_band", observed=True)["abs_error"].agg(n="count", median="median", mean="mean").reindex(band_order))
print("\nsaved reports/figures/06_error_by_segment.png")

erro por property_cluster:
                     n        median           mean
property_cluster                                   
0                 1485   55555.93750   70970.417961
1                 1121   86733.15625  126116.918098
2                   38  269685.78125  323988.392270

erro por price_band:
              n         median           mean
price_band                                   
Entry       771   57353.343750   81341.156615
Standard    821   59401.937500   79596.770421
Premium     570   66616.921875   83661.063596
Luxury      482  100875.875000  172883.627075

saved reports/figures/06_error_by_segment.png


## Notas da extensão — o que isso muda pra engenharia de features

**1. Segmentos são instáveis fora do grupo de fit, e a direção varia por partição (assinatura de
artefato do split, não do mercado):** `price_band` `Luxury` cai de 25% (train, por construção) pra
18,23% em `test` e 17,06% em `val` — direção parecida. Mas `property_cluster` 2 ("luxo físico raro")
**sobe** de 0,75% (train) pra 1,44% em `test`, e **cai** pra 0,31% em `val`. Direções opostas entre
`test`/`val` pro mesmo cluster confirmam a hipótese da fase 03: como o grupo é o zipcode inteiro, a
proporção de um segmento raro numa partição depende de *qual* zipcode caiu ali, não de um viés
sistemático de mercado. Um raw `property_cluster` categórico herda esse ruído.

**2. Mas os segmentos capturam sinal real de erro — a correlação é forte, não coincidência:**
`property_cluster` 2 tem erro médio de **\$313.057** em `test`, contra \$70.283 (cluster 0) e
\$126.049 (cluster 1) — **4,5x pior** que o cluster majoritário. `price_band` `Luxury` tem erro médio
de **\$173.107**, quase o dobro de `Entry` (\$79.565). Isso confirma a hipótese: `property_cluster` é
construído nas mesmas features (`sqft_living`, `bathrooms`, `grade`, `bedrooms`...) que a extensão da
fase 04 já tinha apontado como as que mais correlacionam com erro — o cluster é, na prática, um resumo
dessas features que isola exatamente o grupo mais difícil de prever.

**Conclusão pra fases 07-08:** os dois achados juntos apontam pra uma decisão específica — `property_cluster`
como categoria bruta carrega sinal real (achado 2), mas é instável sob o tipo de split deste projeto
(achado 1). Uma feature **contínua** derivada do mesmo clustering (ex: distância ao centróide mais
próximo, normalizada) preserva o sinal de erro sem depender de um rótulo discreto raro cuja frequência
varia por acaso de qual zipcode caiu em qual partição — o mesmo princípio que já justificou
`comps_knn_price`/`local_grade_percentile` (fase 06/08, KNN espacial contínuo em vez de identidade de
zipcode). `price_band`, por ser ordinal e mais estável entre partições, é candidato mais seguro a entrar
como feature categórica/ordinal direta.

## Artefatos desta extensão

`reports/figures/06_segment_stability_out_of_group.png`, `reports/figures/06_error_by_segment.png` —
ad-hoc, não lidos por nenhum script formal; não alteram `data/trusted/house_segments.parquet` nem
`artifacts/property_cluster_model.pkl`.